# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 5/5 [07:18<00:00, 87.79s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: Senix 21" 60V Cordless Single-Stage Snow Blower for $700 + free shipping\nDetails: That\'s a savings of $59. Buy Now at Walmart\nFeatures: 3,500W brushless motor clears up to 1,200 lbs. of snow per minute throws snow up to 45 feet 21" clearing width and 13" clearing depth chute rotates 200° heated grip reaches up to 110°F for comfort while in use 8" rear wheels built-in LED light bar includes two 60V 8.0Ah batteries and an 8A dual-port fast charger\nURL: https://www.dealnews.com/Senix-21-60-V-Cordless-Single-Stage-Snow-Blower-for-700-free-shipping/21786579.html?iref=rss-c196'

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Refurb Unlocked Apple iPhone 15 Pro Max 256GB Smartphone for $499 + free shipping
Details: That's $38 less than we saw it a few weeks ago, and the best price we've seen. You'd pay $111 more for a refurb at Walmart. Buy Now at eBay
Features: 
URL: https://www.dealnews.com/products/Apple/Unlocked-Apple-iPhone-15-Pro-128-GB-Phone/468789.html?iref=rss-c142

Title: Sennh

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='The Sennheiser Momentum 4 Wireless Headphones combine superior sound quality with exceptional comfort, making them perfect for audiophiles and casual listeners alike. They feature advanced noise cancellation technology, allowing you to immerse yourself in music without distractions. The ergonomic design ensures a snug fit for long listening sessions, while the intuitive touch controls provide seamless navigation. With Bluetooth connectivity and a long battery life, these headphones are an excellent choice for those who value both style and performance.', price=199.95, url='https://www.dealnews.com/Sennheiser-Black-Friday-Sale-Up-to-60-off-free-shipping/21786630.html?iref=rss-c142')

In [3]:
from agents.scanner_agent import ScannerAgent

In [4]:
agent = ScannerAgent()
result = agent.scan()

before parsed
after parsed
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
before parsed
after parsed
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry
processing entry


In [5]:
result

DealSelection(deals=[Deal(product_description='The certified refurbished iRobot Roomba Vac Essential Robot Vacuum is designed to simplify your cleaning routine. It features a 3-stage cleaning system that effectively tackles dirt and debris on both carpets and hardwood floors. With a cleaning schedule that can run for up to 120 minutes per charge, this vacuum ensures a thorough cleaning before returning to its dock. It is especially useful for those who want to maintain cleanliness effortlessly and efficiently.', price=84.0, url='https://www.dealnews.com/products/iRobot/iRobot-Roomba-Vac-Essential-Robot-Vacuum/469604.html?iref=rss-f1912'), Deal(product_description='The Toshiba M550 Series 55M550NU 55" QLED 4K UHD Fire TV offers an immersive viewing experience with its stunning 4K resolution and support for Dolby Vision and HDR technologies. With multiple HDMI inputs and compatibility with smart home devices like Amazon Alexa and Apple HomeKit, it serves as both a television and a smart 